In [1]:
# 1. Install dependencies
!pip install transformers torch nltk numpy spacy

# 2. Download the spaCy small English core model for syntactic parsing
!python -m spacy download en_core_web_sm

# 3. Import the required packages
import math
import torch
import numpy as np
import nltk
import spacy
from transformers import AutoModelForCausalLM, AutoTokenizer

# 4. Download the sentence tokenizer from NLTK
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

# 5. Initialize the spaCy POS parser
nlp = spacy.load("en_core_web_sm")

# 6. Hardware Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 80.1 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Running on: cuda


In [2]:
# Define the model ID
model_id = "openai-community/gpt2"

# Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the Model
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print("Baseline referee loaded and ready.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Baseline referee loaded and ready.


In [3]:
# Known over-indexed vocabulary identified in instruction-tuned outputs
OVERINDEXED_VOCAB = {
    "camaraderie", "tapestry", "intricate", "palpable", "amidst", 
    "solace", "vibrant", "cacophony", "underscore", "unspoken", 
    "fleeting", "unravel", "poignant", "testament", "beacon"
}

# Standard downtoners in the English language
DOWNTONERS = {"barely", "nearly", "almost", "hardly", "scarcely", "partially"}

def analyze_rhetorical_features(text):
    """ENGINE 1: Parses text for grammatical structures heavily favored by instruction tuning."""
    doc = nlp(text)
    
    total_words = len([token for token in doc if token.is_alpha])
    if total_words == 0:
        return {}

    word_lengths = [len(token.text) for token in doc if token.is_alpha]
    
    nominalizations = [
        t.text for t in doc 
        if t.pos_ == "NOUN" and t.text.lower().endswith(("tion", "ment", "ness", "ity", "ance", "ence"))
    ]
    
    present_participles = [t.text for t in doc if t.pos_ == "VERB" and t.tag_ == "VBG"]
    
    infinitives = [t.text for t in doc if t.tag_ == "TO" and t.head.pos_ == "VERB"]
    
    prep_phrases = [t.text for t in doc if t.pos_ == "ADP"]
    
    downtoners = [t.text for t in doc if t.lemma_.lower() in DOWNTONERS]
    
    coordinations = [t.text for t in doc if t.pos_ == "CCONJ"]
    
    found_overindexed = [t.text for t in doc if t.lemma_.lower() in OVERINDEXED_VOCAB]

    return {
        "word_count": total_words,
        "mean_word_length": np.mean(word_lengths) if word_lengths else 0,
        "nominalization_rate": (len(nominalizations) / total_words) * 100,
        "participle_rate": (len(present_participles) / total_words) * 100,
        "infinitive_rate": (len(infinitives) / total_words) * 100,
        "prep_phrase_rate": (len(prep_phrases) / total_words) * 100,
        "downtoner_count": len(downtoners),
        "coordination_rate": (len(coordinations) / total_words) * 100,
        "overindexed_count": len(found_overindexed)
    }

def comprehensive_slop_detector(text):
    """MAIN PIPELINE: Combines Rhetorical Density (Engine 1) with Statistical Physics (Engine 2)."""
    # 1. Run Engine 1: Morphosyntactic Analysis
    rhetorical_metrics = analyze_rhetorical_features(text)
    
    # 2. Run Engine 2: Statistical Perplexity Evaluation
    sentences = sent_tokenize(text)
    sentence_records = []
    
    for sentence in sentences:
        if len(sentence.strip()) < 10:
            continue
            
        inputs = tokenizer(sentence, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            
        perplexity = math.exp(loss.item())
        sentence_records.append({
            "sentence": sentence,
            "perplexity": perplexity
        })
        
    if not sentence_records:
        return None
        
    all_ppl = [record["perplexity"] for record in sentence_records]
    
    return {
        "statistical": {
            "mean_perplexity": np.mean(all_ppl),
            "burstiness": np.std(all_ppl),
            "slop_floor": np.min(all_ppl),
            "sentence_breakdown": sentence_records
        },
        "rhetorical": rhetorical_metrics
    }

In [4]:
sample_passage = """
Navigating the complex landscape of modern organizational dynamics, team members fostered a palpable sense of camaraderie. 
Leaning on their collective strengths, the workforce embarked on the implementation of strategic initiatives, ensuring seamless optimization across departments. 
I watched them from the breakroom, sipping a lukewarm coffee that tasted distinctly like pennies and regret.
"""

# Execute detection routine
analysis = comprehensive_slop_detector(sample_passage)

stat = analysis["statistical"]
rhet = analysis["rhetorical"]

print("=== STATISTICAL VARIANCE (ENGINE 1) ===")
print(f"Overall Perplexity : {stat['mean_perplexity']:.2f}")
print(f"Burstiness Score   : {stat['burstiness']:.2f}")
print(f"Slop Floor         : {stat['slop_floor']:.2f}\n")

print("=== RHETORICAL DENSITY (ENGINE 2) ===")
print(f"Total Words        : {rhet['word_count']}")
print(f"Mean Word Length   : {rhet['mean_word_length']:.1f} chars")
print(f"Nominalizations    : {rhet['nominalization_rate']:.1f}% of text")
print(f"Present Participles: {rhet['participle_rate']:.1f}% of text")
print(f"Over-indexed Words : {rhet['overindexed_count']}\n")

print("=== SENTENCE-LEVEL AUTOPSY ===")
for record in stat['sentence_breakdown']:
    flag = "[LOW VARIANCE]" if record['perplexity'] < 50 else "[HIGH VARIANCE]"
    print(f"{flag} (PPL: {record['perplexity']:.2f}) -> {record['sentence']}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


=== STATISTICAL VARIANCE (ENGINE 1) ===
Overall Perplexity : 94.76
Burstiness Score   : 46.20
Slop Floor         : 43.70

=== RHETORICAL DENSITY (ENGINE 2) ===
Total Words        : 52
Mean Word Length   : 6.4 chars
Nominalizations    : 3.8% of text
Present Participles: 7.7% of text
Over-indexed Words : 2

=== SENTENCE-LEVEL AUTOPSY ===
[LOW VARIANCE] (PPL: 43.70) -> 
Navigating the complex landscape of modern organizational dynamics, team members fostered a palpable sense of camaraderie.
[HIGH VARIANCE] (PPL: 155.59) -> Leaning on their collective strengths, the workforce embarked on the implementation of strategic initiatives, ensuring seamless optimization across departments.
[HIGH VARIANCE] (PPL: 84.98) -> I watched them from the breakroom, sipping a lukewarm coffee that tasted distinctly like pennies and regret.


In [5]:
def classify_text(text):
    """CELL 5: The Final Verdict Engine."""
    analysis = comprehensive_slop_detector(text)
    if not analysis:
        return "ERROR: Text too short for reliable analysis."
        
    stat = analysis["statistical"]
    rhet = analysis["rhetorical"]
    
    slop_score = 0
    flags = []
    
    # 1. Evaluate Statistical Conformity
    if stat["mean_perplexity"] < 60:
        slop_score += 2
        flags.append(f"Low Perplexity ({stat['mean_perplexity']:.1f})")
        
    if stat["burstiness"] < 20:
        slop_score += 1
        flags.append(f"Low Structural Burstiness ({stat['burstiness']:.1f})")
        
    # 2. Evaluate Rhetorical Density
    if rhet["nominalization_rate"] > 4.0:
        slop_score += 1
        flags.append(f"High Nominalization Rate ({rhet['nominalization_rate']:.1f}%)")
        
    if rhet["participle_rate"] > 3.0:
        slop_score += 1
        flags.append(f"High Participle Rate ({rhet['participle_rate']:.1f}%)")
        
    if rhet["overindexed_count"] > 0:
        slop_score += 2
        flags.append(f"Over-indexed Vocabulary Detected")
        
    # 3. The Decision Boundary
    if slop_score >= 4:
        verdict = "HIGH PROBABILITY: AI SLOP"
    elif slop_score >= 2:
        verdict = "MIXED: HEAVILY EDITED HUMAN OR HYBRID"
    else:
        verdict = "HIGH PROBABILITY: HUMAN"
        
    return verdict, flags


human_text = """
As many have come to realize, a human writer’s writing differs largely from AI-generated text. Many factors define human-written content apart from AI-generated “slop”. Some of them are semantic— you just “feel” it. Linguistically, however, some of these factors can be put into math and thus used to identify whether a piece of text was generated by AI or written by a human. Needless to say, this is the basis behind all AI detectors, such as GPTZero or Quillbot, though there can be subtle nuances in methodology.
"""

ai_text = """
Navigating the landscape of AI-generated content, most people can spot machine writing without knowing why, relying on pure instinct. However, a surprising amount of this phenomenon can actually be measured through the evaluation of word choice, sentence rhythm, and grammatical habits, ensuring a palpable distinction between models and humans. This fundamental observation acts as the premise behind detectors like GPTZero or Quillbot, providing seamless optimization of detection methodologies.
"""

print("--- TESTING HUMAN SAMPLE ---")
verdict, flags = classify_text(human_text)
print(f"Verdict: {verdict}")
if flags: print(f"Triggered Flags: {', '.join(flags)}")

print("\n--- TESTING AI SAMPLE ---")
verdict, flags = classify_text(ai_text)
print(f"Verdict: {verdict}")
if flags: print(f"Triggered Flags: {', '.join(flags)}")

--- TESTING HUMAN SAMPLE ---
Verdict: HIGH PROBABILITY: HUMAN

--- TESTING AI SAMPLE ---
Verdict: HIGH PROBABILITY: AI SLOP
Triggered Flags: High Nominalization Rate (8.7%), High Participle Rate (7.2%), Over-indexed Vocabulary Detected
